In [1]:
import pyarrow.parquet as pq
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from python.extraction import extract_filenames
from python.extraction import extract_ranges
from python.extraction import extract_profiles
from python.helpfn_for_prediction import set_index_if_exists
from python.helpfn_for_prediction import adjust_genomic_positions_for_bed
from python.helpfn_for_prediction import move_metadata_columns_to_ranges

import os
import pickle

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)


In [2]:
# directories
working_dir = "/home/zmk214/zmk214/PRIMEloci/evaluation/model_evaluation"
model_dir = "/home/zmk214/zmk214/PRIMEloci_model/2_training/PRIMEloci_GM12878_model_1.0_trvl.sav"
profile_dir = "/home/zmk214/zmk214/PRIMEloci_model/GM12878_wt10M_profiles_te/profiles_subtnorm"

os.chdir(working_dir)

In [3]:
# Load the trained model
with open(model_dir, 'rb') as file:
    model = pickle.load(file)

In [4]:
#Test data
profile_file_ls = extract_filenames(profile_dir)

pos_te_list = [element for element in profile_file_ls if "_pos_" in element]
neg_te_list = [element for element in profile_file_ls if "_neg_" in element]

In [5]:
'''
all_profiles = []

for profile_filename in profile_file_ls:
    filenames_without_extensions = os.path.splitext(profile_filename)[0]
    input_profiles_subtnorm_path = os.path.join(profile_dir, profile_filename)
    subtnorm_df = pd.read_csv(input_profiles_subtnorm_path, header=0, index_col=None)
    subtnorm_df = set_index_if_exists(subtnorm_df)
    subtnorm_np = extract_profiles(subtnorm_df)
    all_profiles.append(subtnorm_np)

if all_profiles:
    all_concatenated_profiles = np.concatenate(all_profiles, axis=0)
    print("Final concatenated array shape:", all_concatenated_profiles.shape)
else:
    print("No profiles were processed.")
'''

'\nall_profiles = []\n\nfor profile_filename in profile_file_ls:\n    filenames_without_extensions = os.path.splitext(profile_filename)[0]\n    input_profiles_subtnorm_path = os.path.join(profile_dir, profile_filename)\n    subtnorm_df = pd.read_csv(input_profiles_subtnorm_path, header=0, index_col=None)\n    subtnorm_df = set_index_if_exists(subtnorm_df)\n    subtnorm_np = extract_profiles(subtnorm_df)\n    all_profiles.append(subtnorm_np)\n\nif all_profiles:\n    all_concatenated_profiles = np.concatenate(all_profiles, axis=0)\n    print("Final concatenated array shape:", all_concatenated_profiles.shape)\nelse:\n    print("No profiles were processed.")\n'

In [6]:

pos_profiles = []

for profile_filename in pos_te_list:
    filenames_without_extensions = os.path.splitext(profile_filename)[0]
    input_profiles_subtnorm_path = os.path.join(profile_dir, profile_filename)
    subtnorm_df = pd.read_csv(input_profiles_subtnorm_path, header=0, index_col=None)
    subtnorm_df = set_index_if_exists(subtnorm_df)
    subtnorm_np = extract_profiles(subtnorm_df)
    pos_profiles.append(subtnorm_np)

if pos_profiles:
    pos_concatenated_profiles = np.concatenate(pos_profiles, axis=0)
    pos_labels  = [1]*len(pos_concatenated_profiles)
    print("Final concatenated array shape:", pos_concatenated_profiles.shape)
else:
    print("No profiles were processed.")


Final concatenated array shape: (300231, 401)


In [7]:

neg_profiles = []

for profile_filename in neg_te_list:
    filenames_without_extensions = os.path.splitext(profile_filename)[0]
    input_profiles_subtnorm_path = os.path.join(profile_dir, profile_filename)
    subtnorm_df = pd.read_csv(input_profiles_subtnorm_path, header=0, index_col=None)
    subtnorm_df = set_index_if_exists(subtnorm_df)
    subtnorm_np = extract_profiles(subtnorm_df)
    neg_profiles.append(subtnorm_np)

if neg_profiles:
    neg_concatenated_profiles = np.concatenate(neg_profiles, axis=0)
    neg_labels  = [0]*len(neg_concatenated_profiles)
    print("Final concatenated array shape:", neg_concatenated_profiles.shape)
else:
    print("No profiles were processed.")


Final concatenated array shape: (300183, 401)


In [8]:
all_concatenated_profiles = np.concatenate((pos_concatenated_profiles, neg_concatenated_profiles), axis=0)
print("Final concatenated array shape:", all_concatenated_profiles.shape)

Final concatenated array shape: (600414, 401)


In [9]:
all_labels = pos_labels + neg_labels
len(all_labels)

600414

In [10]:
test_X = all_concatenated_profiles
test_y = np.array(all_labels)

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.utils import shuffle

# Function to calculate Permutation Feature Importance
def permutation_feature_importance(model, X, y, metric=roc_auc_score, n_repeats=5):
    baseline_score = metric(y, model.predict_proba(X)[:, 1])  # Base performance
    feature_names = [f"Feature_{i}" for i in range(X.shape[1])]  # Map to original indices
    X = pd.DataFrame(X, columns=feature_names)
    importance = pd.DataFrame(index=X.columns, columns=range(n_repeats))

    for col in X.columns:
        for i in range(n_repeats):
            X_permuted = X.copy()
            X_permuted[col] = shuffle(X[col])  # Shuffle feature values
            permuted_score = metric(y, model.predict_proba(X_permuted)[:, 1])
            #print(permuted_score)
            importance.loc[col, i] = baseline_score - permuted_score  # Drop in performance

    return importance.mean(axis=1).sort_values(ascending=False), importance

# Adjust feature indices to -200 to 200
def map_feature_indices(features):
    """Map feature indices from 0-400 to -200 to 200"""
    return [i - 200 for i in range(len(features))]

# Split data by predictions and true labels
print("Split data by predictions and true labels")
pred_probs = model.predict_proba(test_X)[:, 1]
pos_indices_pred = pred_probs >= 0.5
neg_indices_pred = pred_probs < 0.5
#pos_indices_true = test_y == 1
#neg_indices_true = test_y == 0

# Permutation Feature Importance
print("Calculating Permutation Feature Importance for positive predictions...")
importance_pos_pred_mean, importance_pos_pred_repeats = permutation_feature_importance(
    model, test_X[pos_indices_pred], test_y[pos_indices_pred]
)
print("Calculating Permutation Feature Importance for negative predictions...")
importance_neg_pred_mean, importance_neg_pred_repeats = permutation_feature_importance(
    model, test_X[neg_indices_pred], test_y[neg_indices_pred]
)

# Map feature names to -200 to 200
mapped_features = map_feature_indices(importance_pos_pred_mean.index)

# Update index to new mapped features
importance_pos_pred_mean.index = mapped_features
importance_neg_pred_mean.index = mapped_features

# Sort by feature index
importance_pos_pred_mean = importance_pos_pred_mean.sort_index()
importance_neg_pred_mean = importance_neg_pred_mean.sort_index()

# Plot Permutation Feature Importance for Positive Predictions
plt.figure(figsize=(12, 6))
plt.bar(importance_pos_pred_mean.index, importance_pos_pred_mean.values, color='green', alpha=0.7)
plt.title('Permutation Feature Importance (Positive Predictions)')
plt.xlabel('Feature Index (-200 to 200)')
plt.ylabel('Mean Importance (Decrease in AUC)')
plt.grid()
plt.show()

# Plot Permutation Feature Importance for Negative Predictions
plt.figure(figsize=(12, 6))
plt.bar(importance_neg_pred_mean.index, importance_neg_pred_mean.values, color='red', alpha=0.7)
plt.title('Permutation Feature Importance (Negative Predictions)')
plt.xlabel('Feature Index (-200 to 200)')
plt.ylabel('Mean Importance (Decrease in AUC)')
plt.grid()
plt.show()

Split data by predictions and true labels
Calculating Permutation Feature Importance for positive predictions...
Calculating Permutation Feature Importance for negative predictions...


In [ ]:
print("Checking negative prediction importance values...")
print(importance_neg_pred_mean.head(20))  # Display the top 20 values
print("Any non-zero values in negative prediction importance:", (importance_neg_pred_mean != 0).any())


In [ ]:

# SHAP values
print("explainer")
explainer = shap.Explainer(model, test_X)
print("shap_values")
shap_values = explainer(test_X)


In [ ]:
# Visualization of SHAP and Permutation Feature Importance

# Combine SHAP and PFI
def plot_combined_importance(importance, shap_values, subset_indices, title):
    shap_summary = shap_values[subset_indices]
    shap_importance = np.abs(shap_summary.values).mean(axis=0)
    shap_importance_df = pd.Series(shap_importance, index=test_X.columns).sort_values(ascending=False)

    # Combine with PFI
    combined = pd.DataFrame({
        "SHAP Importance": shap_importance_df,
        "PFI Importance": importance.reindex(shap_importance_df.index)
    }).dropna()

    # Plot
    combined.plot(kind="bar", figsize=(12, 6), alpha=0.8)
    plt.title(f"Combined SHAP and PFI: {title}")
    plt.ylabel("Importance")
    plt.xlabel("Features")
    plt.tight_layout()
    plt.savefig(f"combined_importance_{title.replace(' ', '_').lower()}.png")
    plt.show()

# Plot for predicted positives
plot_combined_importance(importance_pos_pred, shap_values, pos_indices_pred, "Predicted Positive")

# Plot for predicted negatives
plot_combined_importance(importance_neg_pred, shap_values, neg_indices_pred, "Predicted Negative")

# Plot for true positives
plot_combined_importance(importance_pos_pred, shap_values, pos_indices_true, "True Positive")

# Plot for true negatives
plot_combined_importance(importance_neg_pred, shap_values, neg_indices_true, "True Negative")